# VocEd Lab 03-v3 — Unsharp Masking

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/emilsar/VocEd/blob/main/03v3_unsharp_masking.ipynb)

Lab 03-v2 introduced a targeted cleanup pipeline:

> **Grayscale → Gaussian blur → NLM denoising → threshold → remove small fragments → fill nucleus holes**

The blur and NLM steps suppress pixel-level noise effectively — but they have a side-effect:
they slightly soften the boundaries between the nucleus and the surrounding cytoplasm.
A softer boundary makes it harder for the grayscale threshold to draw a clean line between
the two classes.

**Unsharp masking** corrects this.  Despite the name, it *sharpens* an image by computing the
difference between the image and a blurred copy of itself and adding that difference back:

$$\text{sharpened} = \text{original} + \alpha \cdot (\text{original} - \text{blurred copy})$$

The term $\text{original} - \text{blurred copy}$ is a *high-pass signal*: it is near zero in
flat, uniform regions (the blur matches the original there) and large at edges (the blur smears
across the boundary, creating a gap).  Multiplying by $\alpha$ — the *amount* — controls how
strongly edges are boosted.  After NLM has cleaned up noise, unsharp masking restores the
crispness of the nucleus boundary without re-introducing the noise.

**New pipeline:**
`Grayscale → Gaussian blur → NLM → unsharp mask → threshold → nucleus cleanup → cytoplasm cleanup`

By the end of this lab you will be able to:
- Visualise the edge-sharpening effect of unsharp masking on a denoised grayscale image.
- Integrate unsharp masking into the segmentation pipeline.
- Tune the sharpening strength $\alpha$ with Bayesian optimisation.
- Compare Dice scores across Labs 01, 02, and 03-v3.


## 0. Setup

In [ ]:
!pip install scikit-optimize scikit-image --quiet

# Clone the repo (skip if already present)
!git clone https://github.com/emilsar/VocEd.git 2>/dev/null || echo 'Already cloned.'
%cd VocEd


## 1. Load Data & Recreate the Train/Test Split

In [ ]:
import glob
import numpy as np
import matplotlib.pyplot as plt
import skimage.filters as skf
import skimage.morphology as skm
import skimage.restoration as skr
from scipy.ndimage import binary_fill_holes
from skopt import gp_minimize
from skopt.space import Real

N = len(glob.glob('imagedata/X/*.npy'))
X = np.stack([np.load(f'imagedata/X/{i}.npy') for i in range(N)])
y = np.stack([np.load(f'imagedata/y/{i}.npy') for i in range(N)])

np.random.seed(42)
idx       = np.random.permutation(N)
train_idx = idx[:160]
test_idx  = idx[160:]

def to_gray(img):
    """Standard ITU-R grayscale from a (3, H, W) float32 image."""
    return 0.299 * img[0] + 0.587 * img[1] + 0.114 * img[2]

def segment_gray(img, t_nuc, t_bg):
    """Lab 01/02 grayscale threshold pipeline — kept here for comparison."""
    gray = to_gray(img)
    pred = np.zeros(gray.shape, dtype=np.int64)
    pred[gray < t_nuc]                    = 2
    pred[(gray >= t_nuc) & (gray < t_bg)] = 1
    return pred

def dice_score(pred, target, cls):
    p = pred   == cls
    t = target == cls
    inter = (p & t).sum()
    denom = p.sum() + t.sum()
    return 1.0 if denom == 0 else 2 * inter / denom

print(f'Loaded {N} images.  Train: {len(train_idx)}  Test: {len(test_idx)}')


## 2. What Unsharp Masking Does

We first convert the RGB image to grayscale, then apply the full preprocessing chain.
After NLM, the grayscale image is smoother — but edges between the nucleus and cytoplasm
are slightly softer.  Unsharp masking recovers that crispness by amplifying the
high-frequency detail NLM suppressed.

Below we compare three stages on image 7:

1. **Grayscale** — the starting point for all subsequent steps.
2. **Grayscale → blur → NLM** — denoised, as in Lab 03-v2.
3. **Grayscale → blur → NLM → unsharp mask** (α = 2.0) — the new step.

The zoomed crop makes the difference easier to see: look at the boundary between the dark
nucleus and the lighter cytoplasm ring.


In [ ]:
IDX = 7

# Step 0: grayscale
gray7 = 0.299 * X[IDX][0] + 0.587 * X[IDX][1] + 0.114 * X[IDX][2]

# Step 1: Gaussian blur on grayscale
blurred = skf.gaussian(gray7, sigma=1.5)

# Step 2: NLM on the blurred grayscale image
sig_est = skr.estimate_sigma(blurred)
nlm     = skr.denoise_nl_means(blurred, h=3 * sig_est,
                                patch_size=5, patch_distance=6,
                                fast_mode=True)

# Step 3: Unsharp masking on the NLM output
sharp = skf.unsharp_mask(nlm, radius=2.0, amount=2.0)
sharp = np.clip(sharp, 0, 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(gray7,  cmap='gray');  axes[0].set_title('Grayscale')
axes[1].imshow(nlm,    cmap='gray');  axes[1].set_title('Grayscale → Blur → NLM')
axes[2].imshow(sharp,  cmap='gray');  axes[2].set_title('Grayscale → Blur → NLM → Unsharp mask  (α = 2.0)')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

# Zoomed crop to highlight edge sharpening
r0, r1, c0, c1 = 80, 160, 80, 160
fig2, axes2 = plt.subplots(1, 3, figsize=(12, 4))
for ax, im, title in zip(axes2,
                          [gray7, nlm, sharp],
                          ['Grayscale (crop)', 'Blur → NLM (crop)', 'Unsharp mask (crop)']):
    ax.imshow(im[r0:r1, c0:c1], cmap='gray')
    ax.set_title(title)
    ax.axis('off')
plt.suptitle('Zoomed in — nucleus/cytoplasm boundary')
plt.tight_layout()
plt.show()


## 3. The Pipeline

`segment_morph_v3` converts to grayscale first, then runs the full preprocessing chain
on that single-channel image before thresholding.  Setting `unsharp_amount=0` turns
sharpening off, recovering the Lab 03-v2 behaviour.

**Parameters for Bayesian optimisation:**

| Parameter | Meaning | Search range |
|---|---|---|
| `t_nuc` | Nucleus grayscale threshold | [0.10, 0.70] |
| `t_bg` | Background grayscale threshold | [0.50, 0.99] |
| `min_nuc_size` | Min nucleus fragment size (pixels) | [10, 500] |
| `min_cyto_size` | Min cytoplasm fragment size (pixels) | [10, 500] |
| `unsharp_amount` | Sharpening strength α | [0.5, 5.0] |


In [ ]:
def segment_morph_v3(img, t_nuc, t_bg, min_nuc_size=50, min_cyto_size=50,
                     blur_sigma=1.5, unsharp_amount=2.0):
    """
    Lab 03-v3 pipeline: grayscale -> blur -> NLM -> unsharp mask ->
                        threshold -> nucleus cleanup -> cytoplasm cleanup.

    img             : (3, H, W) float32, channels-first
    t_nuc           : nucleus threshold (darker pixels -> nucleus)
    t_bg            : background threshold (brighter pixels -> background)
    min_nuc_size    : nucleus fragments smaller than this are removed
    min_cyto_size   : cytoplasm fragments smaller than this are removed
    blur_sigma      : sigma for Gaussian blur
    unsharp_amount  : unsharp mask strength alpha (0 = no sharpening)
    """
    # Stage 0: convert RGB to grayscale
    gray = 0.299 * img[0] + 0.587 * img[1] + 0.114 * img[2]   # (H, W)

    # Pre-processing on grayscale: Gaussian blur -> NLM -> unsharp mask
    gray = skf.gaussian(gray, sigma=blur_sigma)
    sig  = skr.estimate_sigma(gray)
    gray = skr.denoise_nl_means(gray, h=3 * sig,
                                patch_size=5, patch_distance=6,
                                fast_mode=True)
    gray = skf.unsharp_mask(gray, radius=2.0, amount=unsharp_amount)
    gray = np.clip(gray, 0, 1)

    # Stage 1: threshold
    nuc_mask  = gray < t_nuc
    bg_mask   = gray > t_bg
    cyto_mask = ~nuc_mask & ~bg_mask

    # Stage 2: nucleus cleanup
    nuc_mask = skm.remove_small_objects(nuc_mask, min_size=int(min_nuc_size))
    nuc_mask = binary_fill_holes(nuc_mask)

    # Stage 3: cytoplasm cleanup
    if cyto_mask.any():
        cyto_mask = skm.remove_small_objects(cyto_mask, min_size=int(min_cyto_size))

    pred = np.zeros(gray.shape, dtype=np.int64)
    pred[cyto_mask] = 1
    pred[nuc_mask]  = 2
    return pred


pred_v3_7 = segment_morph_v3(X[7], 0.45, 0.85)
print('Classes present:', np.unique(pred_v3_7))


## 4. Visual Comparison on Image 7

In [ ]:
pred_gray7 = segment_gray(X[7], 0.45, 0.85)
pred_v3_7  = segment_morph_v3(X[7], 0.45, 0.85)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].imshow(X[7].transpose(1, 2, 0))
axes[0].set_title('Original RGB')
axes[1].imshow(y[7],       cmap='tab10', vmin=0, vmax=9, interpolation='nearest')
axes[1].set_title('Ground truth')
axes[2].imshow(pred_gray7, cmap='tab10', vmin=0, vmax=9, interpolation='nearest')
axes[2].set_title('Lab 02 \u2014 grayscale threshold')
axes[3].imshow(pred_v3_7,  cmap='tab10', vmin=0, vmax=9, interpolation='nearest')
axes[3].set_title('Lab 03-v3 \u2014 unsharp mask + cleanup')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

for name, pred in [('grayscale', pred_gray7), ('unsharp + cleanup', pred_v3_7)]:
    d1 = dice_score(pred, y[7], 1)
    d2 = dice_score(pred, y[7], 2)
    print(f'Image 7 \u2014 {name:22s}  cytoplasm={d1:.3f}  nucleus={d2:.3f}  mean={0.5*(d1+d2):.3f}')


## 5. Bayesian Optimisation

We search jointly over five parameters: the two grayscale thresholds, the two minimum
fragment sizes, and the unsharp masking strength \u03b1.

The objective is the same as in earlier labs: maximise mean Dice over the 160 training images.
NLM is the slowest step; `fast_mode=True` keeps each evaluation practical.

> **Note:** expect this cell to take **4\u20138 minutes**.


In [ ]:
def objective(params):
    t_nuc, t_bg, min_nuc_size, min_cyto_size, unsharp_amount = params
    scores = []
    for i in train_idx:
        pred = segment_morph_v3(X[i], t_nuc, t_bg,
                                min_nuc_size=min_nuc_size,
                                min_cyto_size=min_cyto_size,
                                unsharp_amount=unsharp_amount)
        scores.append((dice_score(pred, y[i], 1) + dice_score(pred, y[i], 2)) / 2)
    return -np.mean(scores)

search_space = [
    Real(0.10, 0.70, name='t_nuc'),
    Real(0.50, 0.99, name='t_bg'),
    Real(10,   500,  name='min_nuc_size'),
    Real(10,   500,  name='min_cyto_size'),
    Real(0.5,  5.0,  name='unsharp_amount'),
]

print('Running Bayesian optimisation (50 evals)\u2026')
result = gp_minimize(
    func=objective,
    dimensions=search_space,
    n_calls=50,
    n_initial_points=10,
    random_state=42,
    verbose=False,
)
print('Done.')

best_t_nuc, best_t_bg, best_min_nuc, best_min_cyto, best_alpha = result.x
print(f'Best params:  t_nuc={best_t_nuc:.4f}  t_bg={best_t_bg:.4f}  '
      f'min_nuc={best_min_nuc:.0f}  min_cyto={best_min_cyto:.0f}  alpha={best_alpha:.2f}')
print(f'Train Dice:   {-result.fun:.4f}')


## 6. Test-Set Dice Comparison

In [ ]:
# Re-run Lab 02 Bayesian optimisation for a fair grayscale baseline
def objective_gray(params):
    t_nuc, t_bg = params
    scores = [(dice_score(segment_gray(X[i], t_nuc, t_bg), y[i], 1)
             + dice_score(segment_gray(X[i], t_nuc, t_bg), y[i], 2)) / 2
              for i in train_idx]
    return -np.mean(scores)

print('Re-running Lab 02 Bayesian optimisation\u2026')
res02 = gp_minimize(objective_gray,
                    [Real(0.10, 0.70), Real(0.50, 0.99)],
                    n_calls=50, n_initial_points=10, random_state=42, verbose=False)
print('Done.')

# Evaluate on the test set
def test_dice_mean(pred_fn):
    return np.mean([(dice_score(pred_fn(i), y[i], 1) + dice_score(pred_fn(i), y[i], 2)) / 2
                    for i in test_idx])

d_lab01   = test_dice_mean(lambda i: segment_gray(X[i], 0.45, 0.85))
d_lab02   = test_dice_mean(lambda i: segment_gray(X[i], res02.x[0], res02.x[1]))
d_lab03v3 = test_dice_mean(lambda i: segment_morph_v3(X[i], best_t_nuc, best_t_bg,
                                                       min_nuc_size=best_min_nuc,
                                                       min_cyto_size=best_min_cyto,
                                                       unsharp_amount=best_alpha))

print()
print('=' * 60)
print(f'{"Method":<42}  {"Test Dice":>10}')
print('-' * 60)
print(f'{"Lab 01 \u2014 hand-picked grayscale":<42}  {d_lab01:10.4f}')
print(f'{"Lab 02 \u2014 Bayesian opt (grayscale)":<42}  {d_lab02:10.4f}')
print(f'{"Lab 03-v3 \u2014 unsharp mask + Bayesian opt":<42}  {d_lab03v3:10.4f}')
print('=' * 60)


## 7. N/C Ratio Prediction & R\u00b2 vs y\u00a0=\u00a0x

A high Dice score tells us the *shape* of the predicted mask is close to ground truth.
But the clinical question is different: how accurately does our model estimate the
*nucleus-to-cytoplasm (N/C) ratio*?

$$\\text{N/C ratio} = \\frac{\\text{nucleus pixels}}{\\text{cytoplasm pixels}}$$

A high N/C ratio is a hallmark of malignancy.  We evaluate two things:

1. **Scatter plot** \u2014 predicted vs ground-truth ratio.  Points on the diagonal = perfect prediction.
2. **R\u00b2 vs y\u00a0=\u00a0x** \u2014 measures closeness to the identity line, not just rank correlation:

$$R^2 = 1 - \\frac{\\sum_i(\\hat{y}_i - y_i)^2}{\\sum_i(y_i - \\bar{y})^2}$$

$R^2 = 1$ means perfect prediction; $R^2 < 0$ means the predictor is worse than guessing
the mean.


In [ ]:
def nc_ratio(mask):
    """N/C ratio from a (H, W) integer mask.  Returns NaN if cytoplasm is absent."""
    nuc  = (mask == 2).sum()
    cyto = (mask == 1).sum()
    return float('nan') if cyto == 0 else nuc / cyto

def r2_identity(pred, gt_vals):
    """R^2 vs y = x: 1 means perfect agreement with the identity line."""
    ok = ~np.isnan(pred)
    p, g = pred[ok], gt_vals[ok]
    ss_res = np.sum((p - g) ** 2)
    ss_tot = np.sum((g - g.mean()) ** 2)
    return 1 - ss_res / ss_tot

gt_ratios = np.array([nc_ratio(y[i]) for i in test_idx])
valid     = ~np.isnan(gt_ratios)
gt        = gt_ratios[valid]
idx_v     = test_idx[valid]

print(f'Valid test images: {valid.sum()} / {len(test_idx)}')

pred_lab01   = np.array([nc_ratio(segment_gray(X[i], 0.45, 0.85))                          for i in idx_v])
pred_lab02   = np.array([nc_ratio(segment_gray(X[i], res02.x[0], res02.x[1]))              for i in idx_v])
pred_lab03v3 = np.array([nc_ratio(segment_morph_v3(X[i], best_t_nuc, best_t_bg,
                                                    min_nuc_size=best_min_nuc,
                                                    min_cyto_size=best_min_cyto,
                                                    unsharp_amount=best_alpha))             for i in idx_v])

print()
print(f'R\u00b2 (vs y=x) \u2014 Lab 01: {r2_identity(pred_lab01, gt):.4f}   '
      f'Lab 02: {r2_identity(pred_lab02, gt):.4f}   '
      f'Lab 03-v3: {r2_identity(pred_lab03v3, gt):.4f}')


In [ ]:
lim = max(gt.max(),
          np.nanmax(pred_lab01),
          np.nanmax(pred_lab02),
          np.nanmax(pred_lab03v3)) * 1.1

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (title, pred, color) in zip(axes, [
    ('Lab 01 \u2014 hand-picked',     pred_lab01,   'steelblue'),
    ('Lab 02 \u2014 Bayesian',        pred_lab02,   'darkorange'),
    ('Lab 03-v3 \u2014 unsharp mask', pred_lab03v3, 'mediumseagreen'),
]):
    ok = ~np.isnan(pred)
    r2 = r2_identity(pred, gt)
    ax.scatter(gt[ok], pred[ok], alpha=0.6, color=color, s=30)
    ax.plot([0, lim], [0, lim], 'k--', linewidth=1)
    ax.set_xlabel('Ground-truth N/C ratio')
    ax.set_ylabel('Predicted N/C ratio')
    ax.set_title(f'{title}\nR\u00b2 = {r2:.3f}')
    ax.set_xlim(0, lim); ax.set_ylim(0, lim)
    ax.set_aspect('equal')

plt.tight_layout()
plt.show()

print('Points near the dashed diagonal = accurate N/C ratio predictions.')
print('Points scattered far off the line = systematic over- or under-estimation.')


## Wrap-up

**Key takeaways:**

- **Sharpen after denoising, not before.**  Applying unsharp masking to the original
  image would also boost noise.  The correct order is: remove noise first (NLM), then
  restore edge crispness (unsharp mask).

- **\u03b1 is a tunable trade-off.**  A small \u03b1 barely changes the image; a large \u03b1 can
  overshoot and create ringing artefacts (halos around edges).  Bayesian optimisation finds
  the sweet spot for this dataset automatically.

- **Diminishing returns.**  Each lab in this sequence adds one more classical technique.
  The improvement from Lab 02 \u2192 03-v2 \u2192 03-v3 is incremental.  The bottleneck is
  still the grayscale threshold at the core: if two adjacent structures have similar
  brightness, no amount of pre- or post-processing can separate them by thresholding alone.

- **What comes next.**  Labs 04\u201306 move from hand-crafted rules to learned representations:
  pixel classifiers (k-NN), convolutional filters, and finally a full U-Net.  These methods
  learn to distinguish nucleus from cytoplasm using colour and spatial context together,
  bypassing the limitations of a single grayscale threshold.

---

## Group Exercise \u2014 Sharpening Sensitivity

Test three values of `unsharp_amount` on `test_idx[0:5]` using the other optimised parameters.

| Person | Task |
|---|---|
| A | `unsharp_amount = 0.5` (light sharpening) |
| B | `unsharp_amount = 2.0` (moderate sharpening) |
| C | `unsharp_amount = 5.0` (strong sharpening) |

Compare nucleus and cytoplasm Dice for each.  At what point does over-sharpening hurt?
